# k-means clustering

**Objetivo:** agrupar o Iris **sem** usar os rótulos, escolher $k$ pelo cotovelo e pela silhueta, comparar os grupos com as espécies verdadeiras e ver um caso em que o k-means falha.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

iris = load_iris()
X = StandardScaler().fit_transform(iris.data)   # padronizar e essencial
y_verdade = iris.target
print("X:", X.shape, "(rotulos escondidos do algoritmo)")

## 1. Cotovelo e silhueta

A inércia sempre cai com $k$ — procuramos o **cotovelo**. A silhueta média tem um **pico** no $k$ com grupos mais bem separados. Um laço explícito calcula os dois para cada $k$.

In [ ]:
ks = list(range(2, 9))
inercias = []
silhuetas = []
for k in ks:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(X)
    inercias.append(modelo.inertia_)
    silhuetas.append(silhouette_score(X, modelo.labels_))
    print("k =", k, "| inercia", round(modelo.inertia_, 1), "| silhueta", round(silhuetas[-1], 3))

from plotly.subplots import make_subplots
figura = make_subplots(rows=1, cols=2, subplot_titles=("Cotovelo (inercia)", "Silhueta media"))
figura.add_trace(go.Scatter(x=ks, y=inercias, mode="lines+markers", line=dict(color=AZUL)), row=1, col=1)
figura.add_trace(go.Scatter(x=ks, y=silhuetas, mode="lines+markers", line=dict(color=VERDE)), row=1, col=2)
figura.update_layout(height=340, showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
figura.show()
print("k de maior silhueta:", ks[int(np.argmax(silhuetas))])

## 2. Os grupos encontrados × as espécies reais

Com $k=3$, comparamos os grupos do k-means (que nunca viu os rótulos) com as três espécies. A tabela cruzada mostra o quanto eles coincidem.

In [ ]:
modelo = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE).fit(X)
tabela = pd.crosstab(pd.Series(iris.target_names[y_verdade], name="especie real"),
                     pd.Series(modelo.labels_, name="grupo do k-means"))
print(tabela)
print("\nsetosa costuma ficar sozinha num grupo; versicolor e virginica se misturam um pouco.")

## 3. Onde o k-means falha

O k-means supõe grupos **esféricos**. Em dados com formato de duas luas, ele corta pelo meio em vez de seguir as luas — a lição de que a suposição importa.

In [ ]:
from sklearn.datasets import make_moons

X_luas, _ = make_moons(n_samples=300, noise=0.06, random_state=SEMENTE)
grupos_luas = KMeans(n_clusters=2, n_init=10, random_state=SEMENTE).fit_predict(X_luas)

figura = go.Figure(go.Scatter(x=X_luas[:, 0], y=X_luas[:, 1], mode="markers",
                              marker=dict(color=grupos_luas, colorscale="Bluered", size=6)))
figura.update_layout(title="k-means em duas luas: corta reto, ignora a forma",
                     height=380, showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercício

Pela silhueta do item 1, qual $k$ o método sugere para o Iris? Isso bate com o número de espécies? Se não, o que pode explicar a diferença?

<details><summary>Ver resposta</summary>

A silhueta costuma indicar $k=2$ para o Iris, embora existam **três** espécies. O motivo: *setosa* é muito separada das outras duas, enquanto *versicolor* e *virginica* se sobrepõem bastante — do ponto de vista de distância, elas parecem quase um único grupo. A silhueta mede separação geométrica, não conhece as espécies; por isso premia a divisão em 2 grupos bem distintos. É um lembrete de que o "melhor" $k$ estatístico nem sempre é o número de classes reais.

</details>